In [ ]:
!pip install transformers
!pip install timm

In [ ]:
from transformers import DetrImageProcessor, DetrForObjectDetection

image_processor = DetrImageProcessor.from_pretrained(
                      "facebook/detr-resnet-50") 
model = DetrForObjectDetection.from_pretrained(
                      "facebook/detr-resnet-50")

In [ ]:
print(model.config.id2label)

In [ ]:
from PIL import Image, ImageDraw
import requests
from io import BytesIO

def loadImage(url):
    if url.startswith('http'):
        # Fetch the image content and follow redirects
        response = requests.get(url, stream=True, headers={"User-Agent": "Mozilla/5.0"})
        response.raise_for_status()  # Raise error if download failed
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        # Load local image
        image = Image.open(url).convert("RGB")
    return image

image = loadImage(
    "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b9/Good_Smile_Company_offices_ladies.jpg/800px-Good_Smile_Company_offices_ladies.jpg"
)
display(image)

In [ ]:
!pip install Pillow

In [ ]:
inputs = image_processor(images = image, 
                         return_tensors = "pt")

In [ ]:
outputs = model(**inputs)

In [ ]:
print(outputs)

In [ ]:
import torch

target_sizes = torch.tensor([image.size[::-1]])   

results = image_processor.post_process_object_detection(
              outputs,
              target_sizes = target_sizes, 
              threshold = 0.9)[0]
results

In [ ]:
import random

draw = ImageDraw.Draw(image)

for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    print(
        f"Detected {model.config.id2label[label.item()]} with confidence " 
        f"{(score.item() * 100):.2f}% at {box}" 
    )
        
    r = random.randint(0, 255)
    g = random.randint(0, 255)
    b = random.randint(0, 255)
    color = (r, g, b)
    
    draw.rectangle(box,  
                   outline=color,  
                   width=2)  
    
    draw.text((box[0], box[1]-10),  
              model.config.id2label[label.item()],  
              fill='white') 
display(image)

In [ ]:
from transformers import pipeline
 
detection = pipeline("object-detection", model="facebook/detr-resnet-50")

In [ ]:
results = detection(image)
results 

In [ ]:
import random

draw = ImageDraw.Draw(image)
for object in results:
    box = [i for i in object['box'].values()]
    print(
        f"Detected {object['label']} with confidence "  
        f"{(object['score'] * 100):.2f}% at {box}"  
    )    
    r = random.randint(0, 255)
    g = random.randint(0, 255)
    b = random.randint(0, 255)
    color = (r, g, b)    
    draw.rectangle(box,  
                   outline=color,  
                   width=2)    
    draw.text((box[0], box[1]-10), 
              object['label'], 
              fill='white')    
display(image)

In [ ]:
import cv2

stream = cv2.VideoCapture(0) 

while(True):
    (grabbed, frame) = stream.read()
    
    cv2.imshow("Image", frame) 

    key = cv2.waitKey(1) & 0xFF    
    if key == ord("q"):  
        break

stream.release()
cv2.waitKey(1)  
cv2.destroyAllWindows()  
cv2.waitKey(1) 

In [ ]:
from transformers import pipeline
from PIL import Image
import cv2

font   = cv2.FONT_HERSHEY_SIMPLEX  
color  = (0, 255, 255)  # BGR for yellow  
stroke = 2              # thickness for rectangle  

detection = pipeline("object-detection",  
                     model="facebook/detr-resnet-50")  

stream = cv2.VideoCapture(0)  
while(True):    
    (grabbed, frame) = stream.read() 
    image = Image.fromarray(frame)  
    results = detection(image)  
    for object in results:        
        box = [i for i in object['box'].values()]        
        cv2.rectangle(frame,  
                      (box[0],box[1]),  
                      (box[2],box[3]),  
                      color, stroke)
        
        cv2.putText(frame, f'({object["label"]})',  
                    (box[0],box[1]-8), 
                    font, 1, color,  
                    stroke, cv2.LINE_AA)  
    
    cv2.imshow("Image", frame)  
    key = cv2.waitKey(1) & 0xFF    
    if key == ord("q"):  
        break

stream.release() 
cv2.waitKey(1)  
cv2.destroyAllWindows()  
cv2.waitKey(1)  

In [ ]:
from transformers import pipeline

classifier = pipeline("image-classification", 
                      model="ibombonato/vit-age-classifier")

result = classifier(                                                  
    'https://upload.wikimedia.org/wikipedia/commons/' +               
    'thumb/e/e9/Official_portrait_of_Barack_Obama.jpg/' +             
    '800px-Official_portrait_of_Barack_Obama.jpg')                    
print(result)

In [ ]:
from PIL import Image
import requests

from transformers import pipeline

segmentation = pipeline("image-segmentation", 
                model="nvidia/segformer-b0-finetuned-ade-512-512")

segmentation.model.config.id2label

url = 'https://bit.ly/46iDeJQ'
results = segmentation(url)
results

In [ ]:
for result in results:
    print(result['label'])
    display(result['mask'])

In [ ]:
image = Image.open(requests.get(url, stream=True).raw)

for result in results:
    base_image = image.copy()
    mask_image = result['mask']
    
    base_image.paste(mask_image, mask=mask_image)  
    print(result['label'])  
    display(base_image)


In [ ]:
from PIL import ImageOps

for result in results:
    base_image = image.copy()
    mask_image = result['mask']
    
    mask_image = ImageOps.invert(mask_image)  
    base_image.paste(mask_image, mask=mask_image)  
    print(result['label'])  
    display(base_image)


In [ ]:
!pip install gradio

In [ ]:
from transformers import SegformerForSemanticSegmentation
from transformers import pipeline
from PIL import Image, ImageOps

model = pipeline("image-segmentation",                                
                 model="nvidia/segformer-b0-finetuned-ade-512-512")   
    
def segmentation(image, label):    
    image = Image.fromarray(image)                             
    results = model(image)                                     
   
    for result in results:
        if result['label'] == label:
            base_image = image.copy()
            mask_image = result['mask']
            mask_image = ImageOps.invert(mask_image)           
            base_image.paste(mask_image, mask=mask_image)      
            return(base_image)

In [ ]:
import gradio as gr

image_input = gr.Image(label = "Image to segmentize")
label = gr.Textbox(label = "Label to look for", placeholder = "Label")
image_output = gr.Image(label = "Image with the mask applied")

gr.Interface(segmentation,
             [image_input, label],
             image_output).launch()

In [ ]:
!pip install decord

In [ ]:
!pip install eva-decord

In [ ]:
from transformers import pipeline

video_classifier = pipeline("video-classification", 
                   model="MCG-NJU/videomae-base-short-finetuned-kinetics")


In [ ]:
video_classifier.model.config.id2label

In [ ]:
video_classifier(
    'http://localhost:8000/pexels-pat-whelen-5621707 (1080p).mp4')